In [3]:

import pandas as pd
import numpy as np
import pickle
import json

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv('loan_preds.csv')

# Drop ID
df = df.drop('Loan_ID', axis=1)

# Fill categorical with mode
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
df['Married'] = df['Married'].fillna(df['Married'].mode()[0])
df['Dependents'] = df['Dependents'].fillna(df['Dependents'].mode()[0])
df['Self_Employed'] = df['Self_Employed'].fillna(df['Self_Employed'].mode()[0])

# Fill numerical with median
df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].median())

# Credit history
df['Credit_History'] = df['Credit_History'].fillna(df['Credit_History'].mode()[0])

df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})

# feature engineering

df['TotalIncome'] = df['ApplicantIncome'] + df['CoapplicantIncome']
df['IncomeLoanRatio'] = df['TotalIncome'] / df['LoanAmount']

df = pd.get_dummies(df, drop_first=True)

X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)

# train model

model = RandomForestClassifier(n_estimators=200, class_weight='balanced')

model.fit(X_train, y_train)

preds = model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, preds))
print("\nConfusion Matrix:\n", confusion_matrix(y_val, preds))
print("\nClassification Report:\n", classification_report(y_val, preds))

with open('loan_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("\nModel saved successfully!")
print(hasattr(model, "estimators_"))

with open("columns.json", "w") as f:
    json.dump(list(X.columns), f)

Accuracy: 0.7886178861788617

Confusion Matrix:
 [[22 21]
 [ 5 75]]

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.51      0.63        43
           1       0.78      0.94      0.85        80

    accuracy                           0.79       123
   macro avg       0.80      0.72      0.74       123
weighted avg       0.79      0.79      0.77       123


Model saved successfully!
True
